# Mini Accelerator MNIST Demo

This notebook is the planned PYNQ-Z2 demo path. It loads a quantized MNIST artifact, runs a software reference today, and can switch to the FPGA driver once the AXI/MMIO or AXI DMA wrapper exists.

In [ ]:
import sys
from pathlib import Path

repo = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(repo / 'Software' / 'mnist'))

import numpy as np
from accel_format import pack_int8x4
from pynq_demo import software_reference


In [ ]:
artifact = repo / 'artifacts' / 'mnist' / 'mnist_softmax_int8.npz'
data = np.load(artifact)

sample = 0
image = data['sample_images_int8'][sample]
label = int(data['sample_labels'][sample])
weights = data['weights_int8']
bias = data['bias_int8']

logits = software_reference(image, weights, bias)
pred = int(np.argmax(logits))

print('expected:', label)
print('predicted:', pred)
print('packed image words:', pack_int8x4(image.reshape(-1))[:8])


In [ ]:
import matplotlib.pyplot as plt

plt.imshow(image.reshape(28, 28), cmap='gray')
plt.title(f'Expected {label}, predicted {pred}')
plt.axis('off');


## FPGA Hookup TODO

After the AXI wrapper exists, replace the software reference cell with:

```python
from pynq_demo import AcceleratorDriver
driver = AcceleratorDriver('mini_accel.bit')
driver.load_weights(data['weight_words'], data['bias_words'])
logits = driver.run_image(pack_int8x4(image.reshape(-1)))
pred = int(np.argmax(logits))
driver.write_led_digit(pred)
```
